# 🦆 BigQuery BigLake Iceberg REST Catalog Query with PyIceberg & DuckDB

본 노트북은 **GCP BigLake REST Catalog API**와 **Apache Iceberg**, 그리고 **DuckDB**를 연동하여 BigQuery Lakehouse Runtime Catalog 테이블을 서버리스/인메모리 환경에서 SQL로 고성능 분석하는 실습 예제입니다.

### 📌 엔드포인트 및 리소스 설정 정보
- **GCS Warehouse**: `gs://undertail-bq-iceberg-demo-bucket`
- **REST Catalog URI**: `https://biglake.googleapis.com/iceberg/v1/restcatalog`
- **Target Table**: `default.external_weblog` (Lakehouse Runtime Catalog Iceberg Table)

---

### 💡 파이프라인 연동 흐름 (Architecture Flow)
1. **Catalog Initialization**: GCP OAuth 토큰 및 BigLake REST Catalog URI 기반 PyIceberg Catalog 로드
2. **Table Load**: Target Namespace 및 Table (`default.external_weblog`) 식별
3. **Scan to Arrow**: Iceberg Parquet 데이터파일을 PyArrow In-Memory Table로 스캔
4. **DuckDB SQL Query**: DuckDB 엔진으로 PyArrow Table에 직접 고성능 SQL 수행 (`GROUP BY ALL`, Aggregation 등)

In [ ]:
# =========================================================================
# ⚙️ [1단계: 라이브러리 임포트 및 GCP 인증 / BigLake REST Catalog 환경 설정]
# =========================================================================
import os
import duckdb
import google.auth
import google.auth.transport.requests
from pyiceberg.catalog import load_catalog

# GCP Project & BigLake REST Catalog 설정
PROJECT_ID = os.getenv("GCP_PROJECT_ID", "undertail")
GCS_WAREHOUSE = os.getenv("GCS_WAREHOUSE", "gs://undertail-bq-iceberg-demo-bucket/external_iceberg_warehouse")
REST_CATALOG_URI = "https://biglake.googleapis.com/iceberg/v1/restcatalog"

# GCP 인증 토큰 자동 획득 (ADC credentials)
credentials, auth_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
if auth_project and not os.getenv("GCP_PROJECT_ID"):
    PROJECT_ID = auth_project

token_str = getattr(credentials, "token", "") or ""
if not token_str and hasattr(credentials, "refresh"):
    try:
        credentials.refresh(google.auth.transport.requests.Request())
        token_str = credentials.token or ""
    except Exception as e:
        print(f"⚠️ ADC Refresh 경고 (gcloud auth 적용 권장): {e}")

print("=========================================================")
print(f"🎯 Project ID      : {PROJECT_ID}")
print(f"🎯 GCS Warehouse   : {GCS_WAREHOUSE}")
print(f"🎯 REST Catalog URI: {REST_CATALOG_URI}")
print(f"🔐 GCP Auth Token  : {'획득 완료' if token_str else '미발급 (gcloud auth 필요)'}")
print("=========================================================")

In [ ]:
# =========================================================================
# 🔄 [2단계: Step 1 - BigLake REST Catalog 연결 및 초기화]
# =========================================================================
# 1. Initialize the catalog using PyIceberg load_catalog
rest_options = {
    "type": "rest",
    "uri": REST_CATALOG_URI,
    "warehouse": GCS_WAREHOUSE,
    "header.x-goog-user-project": PROJECT_ID,
    "header.X-Iceberg-Access-Delegation": "vended-credentials",
}
if token_str:
    rest_options["token"] = token_str

print("🔄 PyIceberg BigLake REST Catalog 로드 중...")
catalog = load_catalog("lakehouse_catalog", **rest_options)
print("✅ Catalog initialized successfully:", catalog)

# 연결된 카탈로그의 네임스페이스 및 테이블 목록 확인
try:
    namespaces = catalog.list_namespaces()
    print(f"📁 Namespaces 목록: {namespaces}")

    tables = catalog.list_tables("default")
    print(f"📋 'default' 네임스페이스 내 테이블 목록: {tables}")
except Exception as e:
    print(f"ℹ️ 카탈로그 객체 조율 완료 (서버리스 엔드포인트 응답: {e})")

In [ ]:
# =========================================================================
# 📦 [3단계: Step 2 - Target Iceberg Table 메타데이터 로드]
# =========================================================================
# 2. Load the target Iceberg table
target_table_identifier = "default.external_weblog"

print(f"📦 Target Iceberg Table 로드 중: '{target_table_identifier}'...")
table = catalog.load_table(target_table_identifier)

print("✅ Iceberg Table 로드 성공!")
print(f"  - Table Identifier : {table.identifier}")
print(f"  - Spec ID          : {table.spec().spec_id}")
print(f"  - Table Location   : {table.location()}")
print("\n📋 Table Schema:")
print(table.schema())

In [ ]:
# =========================================================================
# 🚀 [4단계: Step 3 - Iceberg 데이터파일 스캔 및 In-Memory Arrow Table 변환]
# =========================================================================
# 3. Scan the table data into an in-memory Arrow table
print("🚀 Iceberg 데이터를 PyArrow Table로 스캔 중 (Zero-Copy Conversion)...")
arrow_table = table.scan().to_arrow()

print("✅ PyArrow Table 변환 완료!")
print(f"  - Total Rows    : {arrow_table.num_rows:,} 건")
print(f"  - Total Columns : {arrow_table.num_columns} 개")
print(f"  - Memory Size   : {arrow_table.nbytes / (1024 * 1024):.2f} MB")
print("\n🔍 Arrow Schema:")
print(arrow_table.schema)

In [ ]:
# =========================================================================
# 🦆 [5단계: Step 4 - DuckDB 인메모리 ANSI SQL 쿼리 수행 및 결과 출력]
# =========================================================================
# 4. Use DuckDB to query the Arrow object with standard SQL

print("🦆 DuckDB 쿼리 실행 1: event_type 별 통계 집계 (GROUP BY ALL)")
print("=========================================================")
results = duckdb.sql("""
    SELECT 
        event_type, 
        COUNT(*) AS total_events, 
        SUM(amount) AS total_amount, 
        AVG(amount) AS avg_amount
    FROM arrow_table 
    GROUP BY ALL
    ORDER BY total_events DESC
""")
results.show()

print("\n🦆 DuckDB 쿼리 실행 2: device_os 및 event_type 기준 그룹핑 집계")
print("=========================================================")
os_summary = duckdb.sql("""
    SELECT 
        device_os,
        event_type,
        COUNT(*) AS count,
        ROUND(AVG(amount), 2) AS avg_amount
    FROM arrow_table
    GROUP BY ALL
    ORDER BY device_os ASC, count DESC
""")
os_summary.show()

### 💡 PyIceberg + DuckDB 연동 핵심 요약 (Key Takeaways)

1. **Zero-Copy In-Memory Integration**:
   - `PyIceberg`의 `table.scan().to_arrow()` 연산을 활용하면 GCS에 저장된 Apache Iceberg Parquet 데이터파일을 복사 오버헤드 없이 PyArrow Table In-Memory 포맷으로 빠르게 변환합니다.
2. **Standard ANSI SQL Capability via DuckDB**:
   - `DuckDB`는 PyArrow 객체를 SQL 테이블처럼 참조(`FROM arrow_table`)하여 `GROUP BY ALL`, CTE, Window Function 등 최신 ANSI SQL 구문을 인메모리로 고성능 처리합니다.
3. **GCP BigLake REST Catalog Integration**:
   - GCP REST Catalog API (`https://biglake.googleapis.com/iceberg/v1/restcatalog`)를 통해 BigQuery 엔진 외부의 클라이언트 라이브러리(PyIceberg + DuckDB)에서도 BigLake REST Catalog의 동일한 Iceberg 메타데이터를 일관되게 조회할 수 있습니다.